# Módulo 1 (a) · Fundamentos de los LLM: generación de texto

**Duración sugerida:** 1.5 horas · **Entrega:** informe con modelo, prompts, resultados y limitaciones.

## Objetivos del laboratorio

- Verificar la instalación de [Ollama](https://ollama.com) y descargar un modelo local.
- Ejecutar la primera generación de texto y entender qué ocurre por dentro.
- Experimentar con **tokens**, **ventana de contexto** y parámetros de generación
  (**temperatura**, `top_p`, `top_k`, `seed`).
- Comparar respuestas deterministas frente a respuestas creativas.

## Conexión con el proyecto ARAS

Este módulo es la **base** de todo el sistema ARAS. Sin entender qué es un LLM,
sus límites y cómo se controla la generación, no es posible diseñar prompts (Módulo 2),
pipelines (Módulo 3) ni agentes (Módulo 4). Aquí fijamos el **modelo local** que
usaremos en el resto del proyecto.

## 1. Verificar Ollama

Ejecuta en la terminal (o en la celda siguiente) el comando que comprueba la versión.

In [ ]:
import subprocess

try:
    out = subprocess.run(["ollama", "--version"], capture_output=True, text=True, timeout=10)
    print("Ollama detectado:", (out.stdout or out.stderr).strip())
except FileNotFoundError:
    print("Ollama NO está instalado. Instálalo desde https://ollama.com y reinicia el kernel.")

## 2. Instalar el cliente Python

Usamos el paquete oficial `ollama`, que se comunica con la API local
(`http://localhost:11434`).

In [ ]:
%pip install -q ollama

## 3. Listar modelos disponibles y descargar uno

Para este curso usaremos `llama3.2` (pequeño y razonablemente capaz en CPU).
Puedes sustituirlo por `qwen2.5:3b`, `mistral` u otro según tu hardware.

In [ ]:
import ollama

MODELO = "llama3.2"

modelos = ollama.list()
print("Modelos locales disponibles:")
for m in modelos.get("models", []):
    size_gb = m.get("size", 0) / 1e9
    print(f"  - {m['name']:24s} {size_gb:.1f} GB")

In [ ]:
import ollama

try:
    ollama.show(MODELO)
    print(f"El modelo '{MODELO}' ya está disponible.")
except Exception:
    print(f"Descargando '{MODELO}' (esto puede tardar unos minutos)...")
    for chunk in ollama.pull(MODELO, stream=True):
        status = chunk.get("status", "")
        if status:
            print(status, end="\r")
    print(f"\n'{MODELO}' descargado.")

## 4. Primera generación de texto

`ollama.generate` envía un `prompt` y devuelve la respuesta del modelo.

In [ ]:
import ollama

resp = ollama.generate(model=MODELO, prompt="Explica qué es un modelo de lenguaje grande en una sola frase.")
print(resp["response"])

## 5. Tokens

Un **token** es la unidad mínima de texto que procesa el modelo (no es exactamente
una palabra: puede ser una subpalabra, un signo o un espacio). Ollama devuelve
cuántos tokens leyó y cuántos generó.

In [ ]:
import ollama

resp = ollama.generate(model=MODELO, prompt="Los tokens son las unidades básicas de texto que procesa un modelo.")

print("Tokens del prompt (entrada):", resp["prompt_eval_count"])
print("Tokens generados (salida):  ", resp["eval_count"])
print("Total de tokens procesados: ", resp["prompt_eval_count"] + resp["eval_count"])

## 6. Parámetros de generación: temperatura

La **temperatura** controla la aleatoriedad:

- `0.0` → determinista (ideal para extracción y respuestas objetivas).
- `~0.7` → equilibrada.
- `>= 1.0` → creativa, pero con más riesgo de incoherencias y alucinaciones.

El parámetro `seed` fija la semilla para que el resultado sea reproducible.

In [ ]:
import ollama

prompt = "Completa la frase: 'El futuro de la inteligencia artificial será...'"

for temp in [0.0, 0.7, 1.4]:
    resp = ollama.generate(
        model=MODELO,
        prompt=prompt,
        options={"temperature": temp, "seed": 42},
    )
    print(f"--- temperatura = {temp} ---")
    print(resp["response"].strip())
    print()

## 7. Otros parámetros: `top_p` y `top_k`

- **`top_p`** (muestreo por núcleo): el modelo elige entre el conjunto más pequeño
  de tokens cuya probabilidad acumulada alcanza `p`.
- **`top_k`**: limita la elección a los `k` tokens más probables.

Ambos reducen la dispersión sin eliminar la variabilidad.

In [ ]:
import ollama

resp = ollama.generate(
    model=MODELO,
    prompt="Nombra tres sinónimos de la palabra 'investigar'.",
    options={"temperature": 0.8, "top_p": 0.5, "top_k": 20},
)
print(resp["response"])

## 8. API de chat con roles

La API de chat permite definir el **rol** de cada mensaje:

- `system`: instrucciones globales de comportamiento.
- `user`: la petición del usuario.
- `assistant`: respuestas previas del modelo (útil en conversaciones).

In [ ]:
import ollama

resp = ollama.chat(
    model=MODELO,
    messages=[
        {"role": "system", "content": "Eres un asistente académico conciso. Responde en una frase."},
        {"role": "user", "content": "Define qué es un agente de software."},
    ],
    options={"temperature": 0.0},
)
print(resp["message"]["content"])

## 9. Ventana de contexto

La **ventana de contexto** es el número máximo de tokens que el modelo puede
"recordar" a la vez. Si la entrada la supera, el modelo pierde información del
principio o la llamada falla. Podemos consultar el límite del modelo y ajustarlo
con `num_ctx`.

In [ ]:
import ollama

info = ollama.show(MODELO)
model_info = info.get("model_info", {})
# El campo exacto puede variar entre versiones de Ollama
for clave in ["llama.context_length", "context_length", "num_ctx"]:
    if clave in model_info:
        print(f"Ventana de contexto de '{MODELO}': {model_info[clave]} tokens")
        break
else:
    print("No se pudo leer la longitud de contexto (revisa la salida de ollama.show).")

In [ ]:
# Demostración: pedir una respuesta muy larga y comprobar el límite de salida.
import ollama

resp = ollama.generate(
    model=MODELO,
    prompt="Escribe un ensayo de 5000 palabras sobre la historia de la computación.",
    options={"num_predict": 80},   # límite de tokens a generar
)
print("Tokens generados (limitado por num_predict):", resp["eval_count"])
print(resp["response"][:300], "...")

## 10. Selección de modelos según la tarea

Criterios prácticos para elegir modelo local:

| Tarea | Característica clave | Ejemplo de modelo |
|-------|----------------------|-------------------|
| Extracción / clasificación | Determinismo, JSON | `llama3.2`, `qwen2.5` |
| Razonamiento y herramientas | *function calling* | `llama3.1`, `qwen2.5` |
| Embeddings (búsqueda) | Calidad del vector | `nomic-embed-text` |
| Hardware limitado | Tamaño reducido | `llama3.2:1b`, `qwen2.5:0.5b` |

> **Regla de oro:** elige el modelo **más pequeño** que resuelva bien tu tarea.
> Menos tamaño = menos latencia, menos memoria y menor coste.

## Resumen del laboratorio

- Configuramos Ollama y descargamos un modelo local.
- Generamos texto y medimos **tokens de entrada/salida**.
- Exploramos **temperatura, `top_p`, `top_k` y `seed`**.
- Usamos la **API de chat con roles**.
- Vimos la **ventana de contexto** y criterios de **selección de modelos**.

**Entregable del Módulo 1:** un breve informe con el modelo usado, ejemplos de
prompts, resultados y limitaciones observadas.